# Christine v2 retrain (TLC) — standalone

Trains **only Christine** on the v2 recipe after v1 collapsed in production
(empty tool calls; all 4 gatekeeper retries failed identically).

What's different from `train_e4b_lora.ipynb`:

- **Christine only.** No Hunter pass — Hunter v1 was healthy and the v2
  hyperparameters (more capacity, more epochs) carry overfit risk for an
  adapter that's already working.
- **`optim='paged_adamw_8bit'`.** The default `adamw_torch` OOMs on T4
  (~14.6 GiB) at v2's r=32 + max_length=8192. 8-bit paged AdamW cuts
  optimizer state ~4× and pages it under pressure.
- **Self-contained.** Each cell can be re-run after the prior ones without
  the cross-cell binding fragility of the original notebook.

Wall-clock estimate: ~3 hours on T4 (Christine has more examples than
Hunter and 5 epochs at 8192 max_length is the slowest config).

**Inputs you provide:**
- `sft_christine.jsonl` (drag into Colab Files panel)
- HF token (read scope is fine)

**Output:** `out/lora-christine/` — the LoRA adapter to merge + quantize
locally.


## 1. Setup — runtime check + deps + HF login

Make sure the runtime is **T4 GPU** (Runtime → Change runtime type → T4 GPU).


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q -U \
    'transformers>=4.50' \
    'peft>=0.13,<0.16' \
    'accelerate>=1.0' \
    'bitsandbytes>=0.44' \
    'datasets>=3.0' \
    'trl>=0.12' \
    'safetensors>=0.4' \
    'sentencepiece' 'protobuf'

import importlib.util, transformers, peft, trl
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('trl:', trl.__version__)
assert importlib.util.find_spec('transformers.models.gemma3') is not None, \
    'no gemma3 in this transformers — pip install transformers --upgrade and restart kernel'


In [ ]:
from huggingface_hub import login
import os

HF_TOKEN = os.environ.get('HF_TOKEN') or input('HF token: ')
login(token=HF_TOKEN)


## 2. Upload Christine's training data

Drag `sft_christine.jsonl` into the Files panel on the left.


In [ ]:
import os
p = '/content/sft_christine.jsonl'
if not os.path.exists(p):
    raise FileNotFoundError(f'{p} not found. Drag sft_christine.jsonl into the Files panel.')
n = sum(1 for _ in open(p))
print(f'sft_christine.jsonl: {n} examples')


## 3. Load Gemma 4 E4B base + tokenizer (8-bit)

8-bit nf4 (not 4-bit) per Selene v7 finding 2026-05-08 — 4-bit produces
unstable Gemma 4 logits that diverge during training.

**SDPA attention** here, not eager. The original notebook used eager
because sdpa was found to silently break softcap during *inference*. But
eager materializes the full attention matrix in fp32 for the softmax
upcast (`[heads × seq × seq] × 4 bytes`), which OOMs T4 even at r=32 +
max_length=4096. SDPA never materializes that matrix. Any softcap
precision drift gets absorbed by the LoRA adapter during training.

bf16 mode skips the GradScaler crashes that fp16 hits with bnb's bf16
activations.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = 'google/gemma-4-e4b-it'
bnb = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('loading base in 8-bit + eager (~3-5 min)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation='sdpa',     # eager OOMs T4 via fp32 softmax temp
)
model.config.use_cache = False

# Disable Gemma 4 thinking-mode attributes — they reserve logic-space for
# <|think|> blocks during training even when the dataset has none.
for attr in ['thinking_mode', 'thinking', 'enable_thinking', 'use_thinking', 'think_before_speak']:
    if hasattr(model.config, attr):
        setattr(model.config, attr, False)
        print(f'  disabled model.config.{attr}')
    if hasattr(model, attr):
        setattr(model, attr, False)
        print(f'  disabled model.{attr}')

print(f'loaded — allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'attn_implementation: {model.config._attn_implementation!r}')
assert model.config._attn_implementation == 'sdpa', 'sdpa attention not engaged'


## 4. Configure LoRA (v2: r=32, alpha=64)

Doubled capacity vs v1 (r=16, alpha=32). v1 collapsed because r=16 was too
small to encode Christine's wider schema (engagement, accommodations,
demos, misconceptions…) without overwriting tool-calling behavior.

Regex `target_modules` scoped to `language_model.layers...` to avoid
training Gemma 4's vision tower on text.


In [ ]:
from peft import LoraConfig

model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

TARGET_MODULES = r'.*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$'

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=TARGET_MODULES,
)


## 5. Train Christine

Self-contained: imports, dataset prep, trainer, train, save — all in this
one cell. Re-runnable: defensive cleanup at the top frees a stale
`trainer` if a previous attempt failed mid-train.

`max_length=4096` here, not the v2 recipe's 8192. Christine's actual
data peaks at ~2400 tokens; the longer context was a phantom fix that
OOMs T4. The real v1 collapse fix is `r=32`.


In [ ]:
import gc
import numpy as np
from datasets import load_dataset
from peft import get_peft_model, PeftModel
from trl import SFTConfig, SFTTrainer

# Free a stale trainer if a previous attempt failed mid-train. Do NOT
# delete `model` — cell 3 loaded it and we want to reuse it.
if 'trainer' in globals():
    del globals()['trainer']
gc.collect()
torch.cuda.empty_cache()

# Gemma chat template prep — fold system into first user, rename
# assistant→model. Gemma's template only knows "user"/"model".
def _prep(ex):
    msgs = ex['messages']
    sys_txt = ''
    out = []
    for m in msgs:
        if m['role'] == 'system':
            sys_txt = m['content'].strip()
        elif m['role'] == 'user':
            content = (sys_txt + '\n\n' + m['content']) if sys_txt else m['content']
            out.append({'role': 'user', 'content': content})
            sys_txt = ''
        elif m['role'] == 'assistant':
            out.append({'role': 'model', 'content': m['content']})
        else:
            out.append(m)
    return {'text': tokenizer.apply_chat_template(out, tokenize=False, add_generation_prompt=False)}

ds = load_dataset('json', data_files='/content/sft_christine.jsonl', split='train')
print('christine examples:', len(ds))
ds = ds.map(_prep, remove_columns=ds.column_names)

# Token-length sanity. v2 originally used max_length=8192 on the theory
# that v1 collapsed due to truncation, but inspection shows Christine's
# data peaks well under 4096 — the real v1 fix is r=32, not the longer
# context. 8192 + r=32 OOMs T4; 4096 fits with headroom.
lengths = [len(tokenizer(x['text']).input_ids) for x in ds]
median_len = int(np.median(lengths))
p90_len = int(np.percentile(lengths, 90))
max_len = max(lengths)
print(f'christine token lengths — median {median_len}, p90 {p90_len}, max {max_len}')
if max_len > 4096:
    over = sum(1 for L in lengths if L > 4096)
    print(f'  ⚠ {over} examples exceed max_length=4096 and will be truncated')
    print('     bump max_length below if this number is meaningful')

# Wrap with PEFT — but only once. Re-running this cell after a successful
# wrap would double-wrap and break the trainable param count.
if not isinstance(model, PeftModel):
    model = get_peft_model(model, peft_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'trainable: {trainable:.1f} M')

cfg = SFTConfig(
    output_dir='out/lora-christine',
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    optim='paged_adamw_8bit',     # 'adamw_torch' OOMs T4 even at 4096
    bf16=True,
    fp16=False,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    # Christine's actual data: median 2002, p90 2188, max 2407 tokens.
    # The original v2 recipe set this to 8192 on the theory that v1 collapsed
    # due to truncation, but the real cause was r=16 being too small for her
    # schema. 4096 fits her data with 70% headroom and stays inside T4's
    # ~14.6 GiB envelope at r=32 with paged 8-bit AdamW.
    max_length=4096,
    packing=False,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds,
    args=cfg,
)

trainer.train()
trainer.save_model('out/lora-christine')
print('\n✓ training complete; adapter saved to out/lora-christine')


## 6. Post-train sanity check — catch collapse before downloading

v1 saved an adapter that produced empty tool calls in production. This
cell generates from a held-out topic and flags suspected collapse so you
don't waste time downloading 200 MB of dead weights.


In [ ]:
print('--- post-train sanity check ---')
test_msg = [{'role': 'user', 'content':
    'Topic: Photosynthesis\n'
    'Grade level: 5th grade\n'
    'Class length: 45 minutes\n'
    'Subject: Science\n\n'
    'No teacher-provided source material. Use your general knowledge; '
    'label all sections with source_origin="not_applicable" (where '
    'appropriate) or "generated".'}]
prompt = tokenizer.apply_chat_template(test_msg, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
trainer.model.eval()
with torch.no_grad():
    out = trainer.model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        repetition_penalty=1.05,
    )
raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('first 1500 chars of generation:\n')
print(raw[:1500])

print('\n--- collapse check ---')
print(f'output length: {len(raw)} chars')
brace_open = raw.count('{')
brace_close = raw.count('}')
print(f'JSON braces: {brace_open} open, {brace_close} close')
has_engagement = '"engagement"' in raw or 'engagement:' in raw.lower()
has_objective = '"objective"' in raw or 'objective:' in raw.lower()
print(f'mentions engagement: {has_engagement}')
print(f'mentions objective: {has_objective}')
if len(raw) < 500 or brace_open < 5:
    print('\n⚠ SUSPECTED COLLAPSE — adapter saved but generation looks '
          'degenerate. Inspect output above before downloading.')
else:
    print('\n✓ output looks healthy at the surface level. Adapter ready '
          'for download.')


## 7. Push to HF Hub (recommended) — survives Colab disconnects

Edit `HF_USER` if your HF username differs.


In [ ]:
from huggingface_hub import HfApi
api = HfApi()

HF_USER = 'hardcoded74'
repo_id = f'{HF_USER}/tlc-gemma-4-e4b-christine-lora'
api.create_repo(repo_id, private=False, exist_ok=True)
api.upload_folder(
    folder_path='out/lora-christine',
    repo_id=repo_id,
    repo_type='model',
)
print(f'pushed {repo_id}')


## 8. Or zip + download (single-file, watch the disconnect timer)


In [ ]:
!zip -r out/lora-christine.zip out/lora-christine
from google.colab import files
files.download('out/lora-christine.zip')


## 9. Next: merge + GGUF locally

Run `training/merge_and_quantize.sh` on Sam's box (where llama.cpp is
built) to merge the LoRA into the base, convert to GGUF, quantize to
Q5_K_M. Drop the result in `~/models/` and the local backend will hot-swap
it for the Christine persona.
